In [ ]:
import numpy as np
import pandas as pd
from scipy.spatial.distance import cosine
from scipy.stats import zscore
import matplotlib.pyplot as plt
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

class Client:
    def __init__(self, client_id, person_id, sensor_data, attack_type='honest'):
        """Initialize client from sensor data"""
        self.id = client_id
        self.person_id = person_id
        self.behavior = attack_type.lower() if attack_type else 'honest'
        self.reputation = 100  # Start with perfect reputation

        # Store sensor data
        self.sensor_data = sensor_data

        # Calculate features from sensor data
        self.accuracy, self.loss = self._calculate_metrics()

        # Generate gradients from sensor data
        self.gradients = self._extract_gradient_features()

        self.timestamp = sensor_data['Time'].iloc[0] if 'Time' in sensor_data.columns else np.random.uniform(0, 1)
        self.ip = f"192.168.{person_id}.{client_id % 255}"

        self.reputation_history = []
        self.suspicious_flags = []
        self.activity_class = sensor_data['Class'].iloc[0] if 'Class' in sensor_data.columns else 'Unknown'

    def _calculate_metrics(self):
        """Calculate accuracy and loss from sensor data"""
        # Use statistical properties of sensor data as proxy metrics

        # Accuracy: Based on consistency of accelerometer readings
        acc_cols = ['Acc_x', 'Acc_y', 'Acc_z']
        acc_variance = self.sensor_data[acc_cols].var().mean()
        accuracy = np.clip(1.0 - (acc_variance / 2.0), 0.0, 1.0)

        # Loss: Based on gyroscope anomaly
        gry_cols = ['Gry_x', 'Gry_y', 'Gry_Z']
        gry_mean_abs = self.sensor_data[gry_cols].abs().mean().mean()
        loss = np.clip(gry_mean_abs / 50.0, 0.0, 5.0)

        # Modify based on attack type
        if self.behavior in ['label_flip', 'label-flip']:
            accuracy = np.random.uniform(0.2, 0.4)
            loss = np.random.uniform(2.0, 4.0)
        elif self.behavior in ['poison', 'model_poison']:
            accuracy = np.random.uniform(0.65, 0.85)
            loss = np.random.uniform(0.3, 0.8)
        elif self.behavior == 'backdoor':
            # Backdoor looks normal but has subtle anomalies
            accuracy = np.random.uniform(0.75, 0.90)
            loss = np.random.uniform(0.2, 0.5)
        elif self.behavior == 'sybil':
            accuracy = np.random.uniform(0.75, 0.85)
            loss = np.random.uniform(0.3, 0.5)

        return accuracy, loss

    def _extract_gradient_features(self):
        """Extract gradient-like features from sensor data"""
        feature_vector = []

        # Extract statistical features from each sensor column
        sensor_cols = ['Acc_x', 'Acc_y', 'Acc_z', 'Gry_x', 'Gry_y', 'Gry_Z']

        for col in sensor_cols:
            if col in self.sensor_data.columns:
                data = self.sensor_data[col].values
                feature_vector.extend([
                    np.mean(data),
                    np.std(data),
                    np.min(data),
                    np.max(data),
                    np.median(data)
                ])

        # Pad or truncate to fixed size (100 dimensions)
        feature_vector = np.array(feature_vector)
        if len(feature_vector) < 100:
            feature_vector = np.pad(feature_vector, (0, 100 - len(feature_vector)))
        else:
            feature_vector = feature_vector[:100]

        # Add noise based on attack type
        if self.behavior in ['poison', 'model_poison']:
            feature_vector *= 5.0  # Poisoned gradients
        elif self.behavior == 'backdoor':
            # Backdoor: subtle manipulation
            feature_vector += np.random.randn(100) * 0.5
        elif self.behavior == 'sybil':
            # Sybil bots have similar features
            np.random.seed(99999)
            feature_vector += np.random.randn(100) * 0.05
            np.random.seed()

        return feature_vector

    def update_metrics(self, round_num):
        """Simulate metric updates over rounds"""
        if self.behavior == 'honest':
            self.accuracy = np.clip(self.accuracy + np.random.uniform(-0.02, 0.03), 0.75, 0.95)
            self.loss = np.clip(self.loss + np.random.uniform(-0.05, 0.05), 0.1, 0.4)
        elif self.behavior in ['label_flip', 'label-flip']:
            self.accuracy = np.clip(self.accuracy + np.random.uniform(-0.05, 0.02), 0.15, 0.45)
            self.loss = np.clip(self.loss + np.random.uniform(-0.1, 0.2), 1.8, 4.0)
        elif self.behavior in ['poison', 'model_poison']:
            self.accuracy = np.clip(self.accuracy + np.random.uniform(-0.03, 0.02), 0.65, 0.85)
            self.loss = np.clip(self.loss + np.random.uniform(-0.05, 0.1), 0.2, 0.8)
        elif self.behavior == 'backdoor':
            self.accuracy = np.clip(self.accuracy + np.random.uniform(-0.01, 0.02), 0.75, 0.90)
            self.loss = np.clip(self.loss + np.random.uniform(-0.03, 0.08), 0.2, 0.6)

        self.gradients = self._extract_gradient_features()
        self.timestamp = round_num + np.random.uniform(0, 0.01)


def load_clients_from_xlsx(file_path, attack_column='attack', samples_per_person=10):
    """Load client data from sensor XLSX file"""
    try:
        df = pd.read_excel(file_path)
        print(f" Loaded {len(df)} sensor readings from {file_path}")
        print(f" Columns found: {list(df.columns)}")

        # Check for attack column
        has_attack_column = attack_column in df.columns

        if not has_attack_column:
            print(f" No '{attack_column}' column found. All clients will be 'honest'.")
            df[attack_column] = 'honest'

        # Group by Person
        clients = []
        client_id = 0

        for person_id, person_data in df.groupby('Person'):
            # Take samples for this person
            num_samples = min(len(person_data), samples_per_person)

            for i in range(num_samples):
                start_idx = i * (len(person_data) // num_samples)
                end_idx = start_idx + 50  # Take 50 readings per client

                if end_idx > len(person_data):
                    end_idx = len(person_data)

                sample_data = person_data.iloc[start_idx:end_idx].copy()

                # Get attack type from first row of sample
                attack_type = sample_data[attack_column].iloc[0] if has_attack_column else 'honest'

                client = Client(
                    client_id=client_id,
                    person_id=int(person_id),
                    sensor_data=sample_data,
                    attack_type=attack_type
                )
                clients.append(client)
                client_id += 1

        # Print statistics
        behavior_counts = {}
        for client in clients:
            behavior_counts[client.behavior] = behavior_counts.get(client.behavior, 0) + 1

        print(f"\n Client Distribution:")
        print(f"  Total Clients: {len(clients)}")
        for behavior, count in behavior_counts.items():
            print(f"  - {behavior}: {count}")

        return clients

    except FileNotFoundError:
        print(f"Error: File '{file_path}' not found!")
        return None
    except Exception as e:
        print(f"Error loading file: {e}")
        import traceback
        traceback.print_exc()
        return None


def compute_median_gradient(all_gradients):
    """Calculate median gradient"""
    gradients_array = np.array(all_gradients)
    return np.median(gradients_array, axis=0)


def cosine_distance(grad1, grad2):
    """Calculate cosine distance"""
    return cosine(grad1, grad2)


def cosine_similarity(grad1, grad2):
    """Calculate cosine similarity"""
    return 1 - cosine(grad1, grad2)


def calculate_norm(gradient):
    """Calculate L2 norm"""
    return np.linalg.norm(gradient)


def check_time_clusters(timestamps, threshold=0.001):
    """Detect timestamp clustering"""
    clustered = []
    for i, t1 in enumerate(timestamps):
        cluster_count = sum(1 for t2 in timestamps if abs(t1 - t2) < threshold)
        if cluster_count > 3:
            clustered.append(t1)
    return clustered


def detect_label_flipping(client, all_clients):
    """Detect label flipping attacks"""
    penalty = 0

    honest_clients = [c for c in all_clients if c.behavior == 'honest']
    if honest_clients:
        avg_accuracy = np.mean([c.accuracy for c in honest_clients])
    else:
        avg_accuracy = 0.85

    if client.accuracy < 0.5:
        penalty += 50
        client.suspicious_flags.append("label_flip_low_acc")

    if client.loss > 2.0:
        penalty += 30
        client.suspicious_flags.append("label_flip_high_loss")

    return penalty


def detect_model_poisoning(client, all_clients):
    """Detect model poisoning attacks"""
    penalty = 0

    all_gradients = [c.gradients for c in all_clients]
    median_gradient = compute_median_gradient(all_gradients)

    distance = cosine_distance(client.gradients, median_gradient)
    if distance > 0.5:
        penalty += 60
        client.suspicious_flags.append("poison_gradient_outlier")

    gradient_norm = calculate_norm(client.gradients)
    average_norm = np.mean([calculate_norm(g) for g in all_gradients])

    if gradient_norm > 3 * average_norm:
        penalty += 40
        client.suspicious_flags.append("poison_large_magnitude")

    return penalty


def detect_sybil_attack(client, all_clients):
    """Detect Sybil attacks"""
    penalty = 0

    # Gradient similarity
    similar_clients = []
    for other_client in all_clients:
        if other_client.id != client.id:
            similarity = cosine_similarity(client.gradients, other_client.gradients)
            if similarity > 0.95:
                similar_clients.append(other_client.id)

    if len(similar_clients) >= 3:
        penalty += 70
        client.suspicious_flags.append("sybil_gradient_similarity")

    # Timestamp clustering
    timestamps = [c.timestamp for c in all_clients]
    time_clusters = check_time_clusters(timestamps)

    if client.timestamp in time_clusters:
        penalty += 20
        client.suspicious_flags.append("sybil_time_cluster")

    # IP clustering
    same_ip_clients = [c for c in all_clients if c.ip == client.ip and c.id != client.id]
    if len(same_ip_clients) >= 2:
        penalty += 30
        client.suspicious_flags.append("sybil_ip_cluster")

    return penalty


def detect_backdoor_attack(client, all_clients):
    """Detect backdoor attacks (subtle anomalies)"""
    penalty = 0

    # Backdoor attacks are subtle - look for inconsistencies
    honest_clients = [c for c in all_clients if c.behavior == 'honest']

    if not honest_clients:
        return penalty

    # Check if client has suspiciously consistent behavior
    if len(client.reputation_history) > 5:
        recent_variance = np.var(client.reputation_history[-5:])
        if recent_variance < 5:  # Too stable = suspicious
            penalty += 30
            client.suspicious_flags.append("backdoor_too_stable")

    # Check for subtle gradient anomalies
    all_gradients = [c.gradients for c in all_clients]
    gradient_stds = [np.std(g) for g in all_gradients]
    client_std = np.std(client.gradients)

    avg_std = np.mean(gradient_stds)
    if abs(client_std - avg_std) > 0.3 * avg_std:
        penalty += 40
        client.suspicious_flags.append("backdoor_gradient_anomaly")

    return penalty


def calculate_reputation_with_attack_detection(client, all_clients, round_num):
    """Main reputation calculation with all attack detection"""
    client.suspicious_flags = []

    # Detect all attack types
    label_flip_penalty = detect_label_flipping(client, all_clients)
    poisoning_penalty = detect_model_poisoning(client, all_clients)
    sybil_penalty = detect_sybil_attack(client, all_clients)
    backdoor_penalty = detect_backdoor_attack(client, all_clients)

    total_penalty = label_flip_penalty + poisoning_penalty + sybil_penalty + backdoor_penalty

    # Update reputation
    new_score = 100 - total_penalty
    client.reputation = 0.7 * new_score + 0.3 * client.reputation
    client.reputation = max(0, min(100, client.reputation))

    client.reputation_history.append(client.reputation)

    return client.reputation


def federated_aggregation(all_clients, threshold=60):
    """Aggregate only trusted clients"""
    trusted_clients = [c for c in all_clients if c.reputation >= threshold]
    excluded_clients = [c for c in all_clients if c.reputation < threshold]

    if len(trusted_clients) == 0:
        return None, [], excluded_clients

    total_weight = sum(c.reputation for c in trusted_clients)
    aggregated_gradient = sum(
        c.gradients * (c.reputation / total_weight)
        for c in trusted_clients
    )

    return aggregated_gradient, trusted_clients, excluded_clients


def run_federated_learning_experiment(clients, num_rounds=30):
    """Run FL experiment with loaded clients"""

    if not clients:
        print("❌ No clients loaded!")
        return None, None

    # Group by behavior
    behavior_groups = {}
    for client in clients:
        behavior = client.behavior
        if behavior not in behavior_groups:
            behavior_groups[behavior] = []
        behavior_groups[behavior].append(client)

    detection_stats = {behavior: [] for behavior in behavior_groups.keys()}
    detection_stats['false_positives'] = []

    print("\n" + "=" * 70)
    print("DREPSEC: FEDERATED LEARNING ATTACK DETECTION ON SENSOR DATA")
    print("=" * 70)
    print(f"Total Clients: {len(clients)}")
    for behavior, group in behavior_groups.items():
        print(f"  - {behavior.upper()}: {len(group)} clients")
    print("=" * 70)

    # Training rounds
    for round_num in range(num_rounds):
        # Update metrics
        for client in clients:
            client.update_metrics(round_num)

        # Calculate reputation
        for client in clients:
            calculate_reputation_with_attack_detection(client, clients, round_num)

        # Aggregate
        result = federated_aggregation(clients, threshold=60)
        if result:
            global_gradient, trusted, excluded = result

        # Calculate detection rates
        for behavior, group in behavior_groups.items():
            if behavior == 'honest':
                detected = [c for c in group if c.reputation < 60]
                detection_stats['false_positives'].append(
                    len(detected) / len(group) * 100 if len(group) > 0 else 0
                )
            else:
                detected = [c for c in group if c.reputation < 60]
                detection_stats[behavior].append(
                    len(detected) / len(group) * 100 if len(group) > 0 else 0
                )

        # Print progress
        if round_num % 5 == 0 or round_num == num_rounds - 1:
            print(f"\n{'='*70}")
            print(f"ROUND {round_num}")
            print(f"{'='*70}")
            for behavior, group in behavior_groups.items():
                if behavior != 'honest':
                    detected = [c for c in group if c.reputation < 60]
                    rate = detection_stats[behavior][-1]
                    print(f"  {behavior.upper():20s}: {len(detected):3d}/{len(group):3d} detected ({rate:5.1f}%)")

            honest_group = behavior_groups.get('honest', [])
            if honest_group:
                false_pos = [c for c in honest_group if c.reputation < 60]
                fp_rate = detection_stats['false_positives'][-1]
                print(f"  {'FALSE POSITIVES':20s}: {len(false_pos):3d}/{len(honest_group):3d} ({fp_rate:5.1f}%)")

            if result:
                print(f"  {'TRUSTED CLIENTS':20s}: {len(trusted):3d}/{len(clients):3d}")

    return clients, detection_stats


def plot_reputation_over_time(clients, save_path='reputation_evolution.png'):
    """Plot reputation scores over time"""

    behavior_groups = {}
    for client in clients:
        behavior = client.behavior
        if behavior not in behavior_groups:
            behavior_groups[behavior] = []
        behavior_groups[behavior].append(client)

    plt.figure(figsize=(14, 8))

    colors = {
        'honest': 'green',
        'label_flip': 'red',
        'label-flip': 'red',
        'poison': 'orange',
        'model_poison': 'orange',
        'backdoor': 'brown',
        'sybil': 'purple'
    }

    linestyles = {
        'honest': '-',
        'label_flip': '--',
        'label-flip': '--',
        'poison': '--',
        'model_poison': '--',
        'backdoor': '-.',
        'sybil': '--'
    }

    for behavior, group in behavior_groups.items():
        if group and len(group[0].reputation_history) > 0:
            rounds = range(len(group[0].reputation_history))
            avg_reputation = np.mean([c.reputation_history for c in group], axis=0)

            color = colors.get(behavior, 'blue')
            linestyle = linestyles.get(behavior, '-')

            plt.plot(rounds, avg_reputation,
                    label=f'{behavior.upper()} ({len(group)} clients)',
                    linewidth=2.5,
                    color=color,
                    linestyle=linestyle)

    plt.axhline(y=60, color='black', linestyle=':', linewidth=2, label='Exclusion Threshold')

    plt.xlabel('Training Round', fontsize=14, fontweight='bold')
    plt.ylabel('Average Reputation Score', fontsize=14, fontweight='bold')
    plt.title('DRepSec: Reputation Evolution on Sensor Data', fontsize=16, fontweight='bold')
    plt.legend(fontsize=11, loc='best')
    plt.grid(True, alpha=0.3)
    plt.ylim(0, 105)

    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"\n Reputation plot saved: {save_path}")
    plt.show()


def plot_detection_rates(detection_stats, save_path='detection_rates.png'):
    """Plot detection rates over time"""

    plt.figure(figsize=(14, 8))

    markers = ['o', 's', '^', 'D', 'v', 'x', 'p']
    marker_idx = 0

    for behavior, rates in detection_stats.items():
        if behavior == 'false_positives':
            plt.plot(range(len(rates)), rates,
                    label='False Positive Rate',
                    linewidth=2.5,
                    marker='x',
                    markersize=5,
                    color='red',
                    linestyle='--')
        elif behavior != 'honest' and len(rates) > 0:
            plt.plot(range(len(rates)), rates,
                    label=f'{behavior.upper()} Detection',
                    linewidth=2.5,
                    marker=markers[marker_idx % len(markers)],
                    markersize=4)
            marker_idx += 1

    plt.xlabel('Training Round', fontsize=14, fontweight='bold')
    plt.ylabel('Detection Rate (%)', fontsize=14, fontweight='bold')
    plt.title('DRepSec: Attack Detection Performance', fontsize=16, fontweight='bold')
    plt.legend(fontsize=12, loc='best')
    plt.grid(True, alpha=0.3)
    plt.ylim(0, 105)

    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"Detection rates plot saved: {save_path}")
    plt.show()


def save_results_to_excel(clients, detection_stats, output_path='drepsec_results.xlsx'):
    """Save results to Excel"""

    # Client results
    client_data = []
    for client in clients:
        client_data.append({
            'client_id': client.id,
            'person_id': client.person_id,
            'behavior': client.behavior,
            'activity_class': client.activity_class,
            'final_reputation': round(client.reputation, 2),
            'initial_accuracy': round(client.accuracy, 3),
            'initial_loss': round(client.loss, 3),
            'was_excluded': client.reputation < 60,
            'suspicious_flags': ', '.join(client.suspicious_flags) if client.suspicious_flags else 'None',
            'ip_address': client.ip
        })

    df_clients = pd.DataFrame(client_data)

    # Detection statistics
    detection_data = []
    for behavior, rates in detection_stats.items():
        if len(rates) > 0:
            detection_data.append({
                'attack_type': behavior,
                'final_detection_rate_%': round(rates[-1], 2),
                'avg_detection_rate_%': round(np.mean(rates), 2),
                'max_detection_rate_%': round(np.max(rates), 2),
                'min_detection_rate_%': round(np.min(rates), 2)
            })

    df_detection = pd.DataFrame(detection_data)

    # Save to Excel
    with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
        df_clients.to_excel(writer, sheet_name='Client Results', index=False)
        df_detection.to_excel(writer, sheet_name='Detection Statistics', index=False)

    print(f"Results saved: {output_path}")


# ========================
# MAIN EXECUTION
# ========================
if __name__ == "__main__":
    import sys

    print("=" * 70)
    print("DRepSec: Blockchain-Based Reputation for Federated Learning")
    print("Sensor Data Attack Detection System")
    print("=" * 70)

    # Get file path
    if len(sys.argv) > 1:
        xlsx_file = sys.argv[1]
    else:
        xlsx_file = input("\nEnter sensor data XLSX file path: ").strip()

    # Load clients
    print(f"\n Loading sensor data from: {xlsx_file}")
    clients = load_clients_from_xlsx(xlsx_file, attack_column='attack', samples_per_person=10)

    if clients:
        # Run experiment
        print("\n Starting Federated Learning experiment...")
        clients, detection_stats = run_federated_learning_experiment(clients, num_rounds=30)

        if clients and detection_stats:
            # Generate visualizations
            print("\n Generating visualizations...")
            plot_reputation_over_time(clients, save_path='reputation_evolution.png')
            plot_detection_rates(detection_stats, save_path='detection_performance.png')

            # Save results
            save_results_to_excel(clients, detection_stats, output_path='drepsec_results.xlsx')

            # Print final summary
            print("\n" + "=" * 70)
            print("FINAL DETECTION RESULTS")
            print("=" * 70)
            for behavior, rates in detection_stats.items():
                if len(rates) > 0:
                    print(f"  {behavior.upper():20s}: {rates[-1]:5.1f}%")
            print("=" * 70)
            print("\n Experiment completed successfully!")
            print(f" Check outputs:")
            print(f"   - reputation_evolution.png")
            print(f"   - detection_performance.png")
            print(f"   - drepsec_results.xlsx")
    else:
        print("\n Failed to load clients. Please check your file format.")

**Attack values you can use:**

honest - Normal client

label_flip - Label flipping attack

poison - Model poisoning attack

sybil - Sybil attack

backdoor - Backdoor attack


**Example:**
Timestamp        Acc_x    Acc_y    Acc_z    Gry_x    Gry_y    Gry_Z    Person  Class   attack

1.56E+12    0.4997   0.7985  -0.3131  -16.768  -10.091   11.189   4       Eating  honest

1.56E+12    0.4973   0.8261  -0.3126   -6.981  -11.310   10.731   4       Eating  honest

1.56E+12    0.4956   0.8255  -0.3134   -5.823  -11.402    3.993   4       Eating  poison

1.56E+12    0.4934   0.8232  -0.3146   -5.091  -11.067    6.890   4       Eating  sybil


**🚀 How to Run:**
# Method 1: Command line
python drepsec_sensor.py your_sensor_data.xlsx

## Method 2: Interactive
python drepsec_sensor.py
## Then enter file path when prompted

**📤 Outputs Generated:**

reputation_evolution.png - Shows how reputation changes for each attack type

detection_performance.png - Detection rates over training rounds

drepsec_results.xlsx - Excel file with:

**Sheet 1:** Each client's final reputation and flags

**Sheet 2:** Detection statistics per attack type

# This code will give us:

```
✅ Real sensor data processing
✅ Multiple attack type detection (label flip, poison, sybil, backdoor)
✅ Publication-quality graphs
✅ Quantitative metrics (detection rates, false positives)
✅ Per-client analysis with suspicious flags
```

